### Assignment: Real-Time Data Analytics with Apache Spark and PostgreSQL

#### Overview

In this assignment, you will perform real-time analytics on streaming data from our music streaming service. The data is streamed into Kafka and includes event information about songs played by different users. Your task is to join this streaming data with static data from our MusicStore PostgreSQL database to perform analytics and create playlists based on specific criteria.

#### Objectives

1. **Join Streaming Data with Static Data**: Use Apache Spark to join real-time streaming data from Kafka with static data from the MusicStore PostgreSQL database.
2. **Create Genre-Based Playlists**: Based on the joined data, generate three playlists for three different genres (Pop, Jazz, Rock), naming each playlist as "Top 20 [Genre]".
3. **Gender-Specific Playlists**: Generate two additional playlists based on the top 20 tracks favored by male and female listeners respectively.

#### Requirements



- **Database Access**: Use the following PostgreSQL credentials to connect to the MusicStore database where static data about tracks, genres, and users are stored.
  - **URL**: `jdbc:postgresql://fhtw-big-data.postgres.database.azure.com:5432/music_store`
  - **User**: `student`
  - **Password**: `reRZ2pjg1WxqlwjU` 
  - **Schema**: `streaming_data`
- **Environment Setup**: Ensure your Spark session is configured to handle both JDBC and Kafka operations.

#### Tasks

- **Join the real-time event data from Kafka with the static track and genre data from the MusicStore database**. Ensure the streaming data schema is properly defined to match the incoming Kafka JSON structure.

##### Task 1: Create Genre-Based Playlists

For at least three of the existing genres (Pop, Jazz, Rock), perform the following:

- Aggregate the top 20 most played tracks within the genre based on real-time streaming data.
- Save the results in a PostgreSQL table named according to your studentID/group_name and the genre, e.g., `DS23M003_playlist_pop`.

##### Task 2: Gender-Specific Playlists

Analyze the streaming data to determine the top 20 tracks for male and female listeners. Create two separate playlists:

- `DS23M003_playlist_male`
- `DS23M003_playlist_female`
- one genre of choice + male or female, eg `ds23m003_playlist_male_rock`

These playlists should reflect the preferences of male and female listeners respectively, based on the frequency of plays.

##### Task 3: Create Top 20 Albums

Based on the track information, create a list of the top 20 albums that these tracks originate from. Perform the following:

- Aggregate the top 20 albums based on the number of tracks played from each album using the real-time streaming data.
- Save the results in a PostgreSQL table named according to your studentID//roup_name and the task name, e.g., `DS23M003_top_20_albums`.

The table should include the following columns:
- **album_id**: The unique identifier for the album.
- **album_name**: The name of the album.
- **artist_name**: The name of the artist associated with the album.

This task will help you understand how to analyze and aggregate streaming data to identify the most popular albums based on user activity in real-time.

### Table Structure for Playlists

For the playlists you are creating, each table should contain the following columns:

- **`track_id`**: The unique identifier for the track.
- **`track_name`**: The name of the track.
- **`artist_name`**: The name of the artist.

Here's a suggested SQL snippet for creating such a table, assuming the data is available from the join operations:

```sql
CREATE TABLE ds23m003_playlist_male (
    track_id INT,
    track_name VARCHAR(255),
    artist_name VARCHAR(255)
);
````

#### Submission Guidelines

- **Code**: Submit a Jupyter Notebook containing all your Spark SQL queries and data processing scripts.

## Solution

### Config Setup

In [1]:
import os

# Path to the PostgreSQL JDBC jar file
rel_path = os.path.relpath('./libs/postgresql-42.7.3.jar')

# Connection URL and properties for PostgreSQL
url = "jdbc:postgresql://fhtw-big-data.postgres.database.azure.com/music_store"
postgres_options = {
    "url": url,
    "user": "student",
    "password": "reRZ2pjg1WxqlwjU",
    "driver": "org.postgresql.Driver"
}

In [2]:
# Kafka configuration parameters for Confluent Cloud
kafka_params = {
    "kafka.bootstrap.servers": "46.225.20.89:9092",
    "subscribe": "music-fhtw",
    "kafka.security.protocol": "PLAINTEXT",
    "startingOffsets": "latest",
}

In [3]:
import os
from pyspark.sql import SparkSession

# Create a Spark session and configure it to use the JDBC jar
spark = SparkSession.builder \
    .appName("PostgreSQL Music Stream Assignment") \
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.4.0") \
    .config("spark.jars", rel_path) \
    .config("spark.executor.extraClassPath", rel_path) \
    .config("spark.driver.extraClassPath", rel_path) \
    .getOrCreate()

26/06/21 21:03:56 WARN Utils: Your hostname, codespaces-f2eebe resolves to a loopback address: 127.0.0.1; using 10.0.14.181 instead (on interface eth0)
26/06/21 21:03:56 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/usr/local/sdkman/candidates/spark/3.5.1/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/vscode/.ivy2/cache
The jars for the packages stored in: /home/vscode/.ivy2/jars
org.apache.spark#spark-sql-kafka-0-10_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-eb550244-c039-4733-963b-0b621258775c;1.0
	confs: [default]
	found org.apache.spark#spark-sql-kafka-0-10_2.12;3.4.0 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.12;3.4.0 in central
	found org.apache.kafka#kafka-clients;3.3.2 in central
	found org.lz4#lz4-java;1.8.0 in central
	found org.xerial.snappy#snappy-java;1.1.9.1 in central
	found org.slf4j#slf4j-api;2.0.6 in central
	found org.apache.hadoop#hadoop-client-runtime;3.3.4 in central
	found org.apache.hadoop#hadoop-client-api;3.3.4 in central
	found commons-logging#commons-logging;1.1.3 in central
	found com.google.code.findbugs#jsr305;3.0.0 in central
	found org.apache.commons#commons-pool2;2.11.1 in central
:: resolution report :: resolve 884ms :: artifacts dl 32ms
	:: 

### Parsing Schema

In [4]:
from pyspark.sql.types import StructType, StructField, StringType, LongType, IntegerType

# Define the schema for the JSON data from Kafka
json_schema = StructType([
    StructField("ts", LongType(), True),
    StructField("auth", StringType(), True),
    StructField("page", StringType(), True),
    StructField("song", StringType(), True),
    StructField("level", StringType(), True),
    StructField("artist", StringType(), True),
    StructField("gender", StringType(), True),
    StructField("method", StringType(), True),
    StructField("status", IntegerType(), True),
    StructField("userId", StringType(), True),
    StructField("lastName", StringType(), True),
    StructField("location", StringType(), True),
    StructField("track_id", IntegerType(), True),
    StructField("firstName", StringType(), True),
    StructField("sessionId", IntegerType(), True),
    StructField("userAgent", StringType(), True),
    StructField("registration", LongType(), True),
    StructField("itemInSession", IntegerType(), True)
])

In [5]:
from pyspark.sql.functions import col, from_json

# Read streaming data from Kafka
df = spark \
    .readStream \
    .format("kafka") \
    .options(**kafka_params) \
    .load()

# Select and cast the key and value columns to STRING
df_message = df.selectExpr("CAST(key AS STRING)", "CAST(value AS STRING)")

In [6]:
# Parse the JSON data and apply schema
df_value = df_message.selectExpr("CAST(value AS STRING) as json_string")
parsed_df = (df_value
    .withColumn("jsonData", from_json(col("json_string"), json_schema))
    .select("jsonData.*")
    )

In [ ]:
# truncate false
query = parsed_df.writeStream \
    .outputMode("append") \
    .format("console") \
    .option("truncate", "false") \
    .start()

import time
time.sleep(10)
query.stop()

26/06/21 21:04:31 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-573ec565-2faf-47d3-862b-e1e7ab3cee10. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/06/21 21:04:31 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
26/06/21 21:04:32 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.


-------------------------------------------
Batch: 0
-------------------------------------------
+---+----+----+----+-----+------+------+------+------+------+--------+--------+--------+---------+---------+---------+------------+-------------+
|ts |auth|page|song|level|artist|gender|method|status|userId|lastName|location|track_id|firstName|sessionId|userAgent|registration|itemInSession|
+---+----+----+----+-----+------+------+------+------+------+--------+--------+--------+---------+---------+---------+------------+-------------+
+---+----+----+----+-----+------+------+------+------+------+--------+--------+--------+---------+---------+---------+------------+-------------+



-------------------------------------------
Batch: 1
-------------------------------------------
+-------------+---------+--------+-----------------------------------------------------------------+-----+---------------------------------------------+------+------+------+------+--------+------------------------------+--------+---------+---------+---------------------------------------------------------------------------------------------------------------+-------------+-------------+
|ts           |auth     |page    |song                                                             |level|artist                                       |gender|method|status|userId|lastName|location                      |track_id|firstName|sessionId|userAgent                                                                                                      |registration |itemInSession|
+-------------+---------+--------+-----------------------------------------------------------------+-----+-----------------

-------------------------------------------
Batch: 2
-------------------------------------------
+-------------+---------+--------+-----------------------------------------------------------------+-----+---------------------------------------------+------+------+------+------+--------+------------------------------+--------+---------+---------+---------------------------------------------------------------------------------------------------------------+-------------+-------------+
|ts           |auth     |page    |song                                                             |level|artist                                       |gender|method|status|userId|lastName|location                      |track_id|firstName|sessionId|userAgent                                                                                                      |registration |itemInSession|
+-------------+---------+--------+-----------------------------------------------------------------+-----+-----------------

-------------------------------------------
Batch: 4
-------------------------------------------
+-------------+---------+--------+--------------------------------+-----+--------------+------+------+------+------+--------+------------------------------+--------+---------+---------+---------------------------------------------------------------------------------------------------------------+-------------+-------------+
|ts           |auth     |page    |song                            |level|artist        |gender|method|status|userId|lastName|location                      |track_id|firstName|sessionId|userAgent                                                                                                      |registration |itemInSession|
+-------------+---------+--------+--------------------------------+-----+--------------+------+------+------+------+--------+------------------------------+--------+---------+---------+------------------------------------------------------------------

-------------------------------------------
Batch: 6
-------------------------------------------
+-------------+---------+--------+--------------------------------+-----+-------------+------+------+------+------+--------+------------------------------+--------+---------+---------+---------------------------------------------------------------------------------------------------------------+-------------+-------------+
|ts           |auth     |page    |song                            |level|artist       |gender|method|status|userId|lastName|location                      |track_id|firstName|sessionId|userAgent                                                                                                      |registration |itemInSession|
+-------------+---------+--------+--------------------------------+-----+-------------+------+------+------+------+--------+------------------------------+--------+---------+---------+---------------------------------------------------------------------

-------------------------------------------
Batch: 8
-------------------------------------------
+-------------+---------+--------+--------------------------------+-----+---------------+------+------+------+------+--------+------------------------------+--------+---------+---------+---------------------------------------------------------------------------------------------------------------+-------------+-------------+
|ts           |auth     |page    |song                            |level|artist         |gender|method|status|userId|lastName|location                      |track_id|firstName|sessionId|userAgent                                                                                                      |registration |itemInSession|
+-------------+---------+--------+--------------------------------+-----+---------------+------+------+------+------+--------+------------------------------+--------+---------+---------+---------------------------------------------------------------

-------------------------------------------
Batch: 10
-------------------------------------------
+-------------+---------+--------+----------------------------+-----+------------------------------+------+------+------+------+--------+------------------------------+--------+---------+---------+---------------------------------------------------------------------------------------------------------------+-------------+-------------+
|ts           |auth     |page    |song                        |level|artist                        |gender|method|status|userId|lastName|location                      |track_id|firstName|sessionId|userAgent                                                                                                      |registration |itemInSession|
+-------------+---------+--------+----------------------------+-----+------------------------------+------+------+------+------+--------+------------------------------+--------+---------+---------+-----------------------------

### Join with Postgres data

In [102]:
df_genres = spark.read.jdbc(url=url, 
                            table="""(
                            SELECT tracks.id AS db_track_id, g.name AS genre 
                            FROM tracks 
                            JOIN public.genres g ON g.id = tracks.genre_id) 
                            AS genre_data
                            """, 
                            properties=postgres_options)

In [103]:
# Join the data and filter to played song
df_joined = parsed_df.join(df_genres, parsed_df.track_id == df_genres.db_track_id, "inner")
df_played = df_joined.filter(col("page") == "NextSong")

In [104]:
# Check available genres
from pyspark.sql.functions import count, desc

df_genres.groupBy("genre").agg(count("*").alias("count")).orderBy(desc("count")).show(20)

+------------------+-----+
|             genre|count|
+------------------+-----+
|              Rock| 1297|
|             Latin|  579|
|             Metal|  374|
|Alternative & Punk|  332|
|              Jazz|  130|
|          TV Shows|   93|
|             Blues|   81|
|         Classical|   74|
|             Drama|   64|
|          R&B/Soul|   61|
|            Reggae|   58|
|               Pop|   48|
|        Soundtrack|   43|
|       Alternative|   40|
|       Hip Hop/Rap|   35|
| Electronica/Dance|   30|
|       Heavy Metal|   28|
|             World|   28|
|  Sci Fi & Fantasy|   26|
|    Easy Listening|   24|
+------------------+-----+
only showing top 20 rows



### Task 1: Create Genre-Based Playlists

For at least three of the existing genres (Pop, Jazz, Rock), perform the following:

- Aggregate the top 20 most played tracks within the genre based on real-time streaming data.
- Save the results in a PostgreSQL table named according to your studentID/group_name and the genre, e.g., `DS23M003_playlist_pop`.

In [27]:
spark.conf.set("spark.sql.shuffle.partitions", "4")

In [ ]:
student_id = "ds25m045"
genre = "Latin"

table_name = f"streaming_data.{student_id}_playlist_{genre}"

In [33]:
# for checking results
def display_stream(df, timeout=10):
    query = df.writeStream \
        .outputMode("complete") \
        .format("console") \
        .option("truncate", "false") \
        .start()
    # Let the stream run for a short period and then stop (for demonstration purposes)
    time.sleep(timeout)
    query.stop()

In [59]:
# Write results to PostgreSQL using foreachBatch
def write_to_postgres(df, epoch_id):
    df.write \
        .format("jdbc") \
        .options(**postgres_options) \
        .option("dbtable", table_name) \
        .mode("overwrite") \
        .save()

#### Latin Playlist

In [53]:
df_latin = (
    df_played
    .filter(col("genre") == genre)
    .groupBy(["track_id", "song", "artist"])
    .agg(count("*").alias("play_count"))
    .orderBy(desc("play_count"))
    .limit(20)
            )

In [56]:
# Testing output
display_stream(df_latin)

26/06/21 22:40:17 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-bf51dd86-6a09-4978-a22c-625636638952. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/06/21 22:40:17 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
26/06/21 22:40:17 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.


-------------------------------------------
Batch: 0
-------------------------------------------
+--------+----+------+----------+
|track_id|song|artist|play_count|
+--------+----+------+----------+
+--------+----+------+----------+



-------------------------------------------
Batch: 1
-------------------------------------------
+--------+----------------------------------+---------------------------+----------+
|track_id|song                              |artist                     |play_count|
+--------+----------------------------------+---------------------------+----------+
|253     |Maracatu Atômico                  |Chico Science & Nação Zumbi|1         |
|507     |Lindo Lago Do Amor                |Gonzaguinha                |1         |
|2083    |Vital E Sua Moto                  |Os Paralamas Do Sucesso    |1         |
|1511    |País Tropical                     |Jorge Ben                  |1         |
|404     |Mr Funk Samba                     |Antônio Carlos Jobim       |1         |
|723     |1° De Julho                       |Cássia Eller               |1         |
|219     |Odara                             |Caetano Veloso             |1         |
|236     |A Banda                           |Chico Bu

-------------------------------------------
Batch: 2
-------------------------------------------
+--------+------------------------------+---------------------------+----------+
|track_id|song                          |artist                     |play_count|
+--------+------------------------------+---------------------------+----------+
|240     |Meu Caro Amigo                |Chico Buarque              |1         |
|739     |Eleanor Rigby                 |Cássia Eller               |1         |
|2751    |Primavera                     |Tim Maia                   |1         |
|985     |Medo De Escuro                |Falamansa                  |1         |
|1741    |Assaltaram A Gramática        |Lulu Santos                |1         |
|3161    |Minha Fé                      |Zeca Pagodinho             |1         |
|2776    |Preciso Ser Amado             |Tim Maia                   |1         |
|729     |Juventude Transviada (Ao Vivo)|Cássia Eller               |1         |
|594     |Ag

26/06/21 22:40:27 ERROR WriteToDataSourceV2Exec: Data source write support MicroBatchWrite[epoch: 3, writer: ConsoleWriter[numRows=20, truncate=false]] is aborting.
26/06/21 22:40:27 ERROR WriteToDataSourceV2Exec: Data source write support MicroBatchWrite[epoch: 3, writer: ConsoleWriter[numRows=20, truncate=false]] aborted.
26/06/21 22:40:27 ERROR ShuffleBlockFetcherIterator: Error occurred while fetching local blocks, null
26/06/21 22:40:27 ERROR TaskContextImpl: Error in TaskCompletionListener
java.io.FileNotFoundException: File file:/tmp/temporary-bf51dd86-6a09-4978-a22c-625636638952/state/0/2 does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:779)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1100)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:769)
	at org.apache.hadoop.fs.DelegateToFileSystem.getFileStatus(DelegateToFileSystem.java:128)
	at or

In [60]:
print(f'Starting to write to table {table_name}')

# Display the count of trades for each symbol
query = df_latin.writeStream \
    .outputMode("complete") \
    .foreachBatch(write_to_postgres) \
    .start()

time.sleep(90)
query.stop()
print("Done")

Starting to write to table streaming_data.ds25m045_playlist_Latin


26/06/21 22:43:07 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-577ed481-6a8f-45e7-aedd-2d9aa6961e1f. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/06/21 22:43:07 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
26/06/21 22:43:07 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.


Done


#### Rock Playlist

In [63]:
genre = "Rock"
table_name = f"streaming_data.{student_id}_playlist_{genre}"

In [62]:
df_rock = (
    df_played
    .filter(col("genre") == genre)
    .groupBy(["track_id", "song", "artist"])
    .agg(count("*").alias("play_count"))
    .orderBy(desc("play_count"))
    .limit(20)
            )

In [64]:
print(f'Starting to write to table {table_name}')

# Display the count of trades for each symbol
query = df_rock.writeStream \
    .outputMode("complete") \
    .foreachBatch(write_to_postgres) \
    .start()

time.sleep(90)
query.stop()
print("Done")

Starting to write to table streaming_data.ds25m045_playlist_Rock


26/06/21 22:47:51 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-2b917e0f-a60d-4266-8de3-2d80db380ffb. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/06/21 22:47:51 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
26/06/21 22:47:51 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.


Done


#### R&B/Soul Playlist

In [68]:
genre = "R&B/Soul"
table_name = f"streaming_data.{student_id}_playlist_rnb"

In [69]:
df_rnb = (
    df_played
    .filter(col("genre") == genre)
    .groupBy(["track_id", "song", "artist"])
    .agg(count("*").alias("play_count"))
    .orderBy(desc("play_count"))
    .limit(20)
            )

In [70]:
print(f'Starting to write to table {table_name}')

# Display the count of trades for each symbol
query = df_rnb.writeStream \
    .outputMode("complete") \
    .foreachBatch(write_to_postgres) \
    .start()

time.sleep(90)
query.stop()
print("Done")

Starting to write to table streaming_data.ds25m045_playlist_rnb


26/06/21 22:54:21 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-a005fb52-0e30-4922-b95f-7342f6089d7f. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/06/21 22:54:21 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
26/06/21 22:54:21 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.


Done


### Task 2: Gender-Specific Playlists

Analyze the streaming data to determine the top 20 tracks for male and female listeners. Create two separate playlists:

- `DS23M003_playlist_male`
- `DS23M003_playlist_female`
- one genre of choice + male or female, eg `ds23m003_playlist_male_rock`

These playlists should reflect the preferences of male and female listeners respectively, based on the frequency of plays.

#### Playlist Male

In [71]:
gender = "M"
table_name = f"streaming_data.{student_id}_playlist_male"

In [78]:
df_male = (
    df_played
    .filter(col("gender") == gender)
    .groupBy(["track_id", "song", "artist", "gender"])
    .agg(count("*").alias("play_count"))
    .orderBy(desc("play_count"))
    .limit(20)
            )

In [79]:
print(f'Starting to write to table {table_name}')

# Display the count of trades for each symbol
query = df_male.writeStream \
    .outputMode("complete") \
    .foreachBatch(write_to_postgres) \
    .start()

time.sleep(90)
query.stop()
print("Done")

Starting to write to table streaming_data.ds25m045_playlist_male


26/06/21 23:00:58 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-bdbfb427-5898-4db5-81d6-d60f298bc580. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/06/21 23:00:58 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
26/06/21 23:00:59 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.


Done


#### Playlist Female

In [ ]:
gender = "F"

In [81]:
df_female = (
    df_played
    .filter(col("gender") == gender)
    .groupBy(["track_id", "song", "artist", "gender"])
    .agg(count("*").alias("play_count"))
    .orderBy(desc("play_count"))
    .limit(20)
            )

In [ ]:
table_name = f"streaming_data.{student_id}_playlist_female"

print(f'Starting to write to table {table_name}')

# Display the count of trades for each symbol
query = df_female.writeStream \
    .outputMode("complete") \
    .foreachBatch(write_to_postgres) \
    .start()

time.sleep(90)
query.stop()
print("Done")

Starting to write to table streaming_data.ds25m045_playlist_female


26/06/21 23:02:40 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-49d695d6-494a-4c52-b00d-e5a09d4208ad. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/06/21 23:02:40 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
26/06/21 23:02:40 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.


Done


#### Playlist Male + R&B

In [108]:
gender = "M"
genre = "R&B/Soul"

In [109]:
df_male_rnb = (
    df_played
    .filter((col("gender") == gender) & (col("genre") == genre))
    .groupBy(["track_id", "song", "artist", "gender", "genre"])
    .agg(count("*").alias("play_count"))
    .orderBy(desc("play_count"))
    .limit(20)
            )

In [110]:
table_name = f"streaming_data.{student_id}_playlist_male_rnb"

print(f'Starting to write to table {table_name}')

# Display the count of trades for each symbol
query = df_male_rnb.writeStream \
    .outputMode("complete") \
    .foreachBatch(write_to_postgres) \
    .start()

time.sleep(90)
query.stop()
print("Done")

Starting to write to table streaming_data.ds25m045_playlist_male_rnb


26/06/21 23:32:39 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-99274877-5d91-484e-a541-c779a5527f1d. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/06/21 23:32:39 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
26/06/21 23:32:39 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.


Done


26/06/21 23:34:09 ERROR TorrentBroadcast: Store broadcast broadcast_3830 fail, remove all pieces of the broadcast


### Task 3: Create Top 20 Albums

Based on the track information, create a list of the top 20 albums that these tracks originate from. Perform the following:

- Aggregate the top 20 albums based on the number of tracks played from each album using the real-time streaming data.
- Save the results in a PostgreSQL table named according to your studentID//roup_name and the task name, e.g., `DS23M003_top_20_albums`.

The table should include the following columns:
- **album_id**: The unique identifier for the album.
- **album_name**: The name of the album.
- **artist_name**: The name of the artist associated with the album.

This task will help you understand how to analyze and aggregate streaming data to identify the most popular albums based on user activity in real-time.

In [95]:
df_albums = spark.read.jdbc(url=url, 
                            table="""(
                            SELECT tracks.id AS db_track_id, a.title AS album_name, a.id AS album_id
                            FROM tracks 
                            JOIN public.albums a ON a.id = tracks.album_id) 
                            AS genre_data
                            """, 
                            properties=postgres_options)

In [96]:
# Join the data and filter to played song
df_joined_album = parsed_df.join(df_albums, parsed_df.track_id == df_albums.db_track_id, "inner")
df_played_album = df_joined_album.filter(col("page") == "NextSong")

In [97]:
df_albums_top20 = (
    df_played_album
    .groupBy(["album_id", "album_name", "artist"])
    .agg(count("*").alias("play_count"))
    .orderBy(desc("play_count"))
    .limit(20)
            )

In [98]:
table_name = f"streaming_data.{student_id}_top_20_albums"

print(f'Starting to write to table {table_name}')

# Display the count of trades for each symbol
query = df_albums_top20.writeStream \
    .outputMode("complete") \
    .foreachBatch(write_to_postgres) \
    .start()

time.sleep(90)
query.stop()
print("Done")

Starting to write to table streaming_data.ds25m045_top_20_albums


26/06/21 23:23:51 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-68884bf8-eee7-4c10-af7c-9a6e9c35a89d. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/06/21 23:23:51 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
26/06/21 23:23:51 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.


Done


In [ ]:
# stop all running spark streams
for s in spark.streams.active:
    s.stop()

spark.stop()

26/06/21 23:35:08 WARN StateStore: Error running maintenance thread
java.lang.IllegalStateException: SparkEnv not active, cannot do maintenance on StateStores
	at org.apache.spark.sql.execution.streaming.state.StateStore$.doMaintenance(StateStore.scala:632)
	at org.apache.spark.sql.execution.streaming.state.StateStore$.$anonfun$startMaintenanceIfNeeded$1(StateStore.scala:610)
	at org.apache.spark.sql.execution.streaming.state.StateStore$MaintenanceTask$$anon$1.run(StateStore.scala:453)
	at java.base/java.util.concurrent.Executors$RunnableAdapter.call(Executors.java:539)
	at java.base/java.util.concurrent.FutureTask.runAndReset(FutureTask.java:305)
	at java.base/java.util.concurrent.ScheduledThreadPoolExecutor$ScheduledFutureTask.run(ScheduledThreadPoolExecutor.java:305)
	at java.base/java.util.concurrent.ThreadPoolExecutor.runWorker(ThreadPoolExecutor.java:1136)
	at java.base/java.util.concurrent.ThreadPoolExecutor$Worker.run(ThreadPoolExecutor.java:635)
	at java.base/java.lang.Thread.